# PC Specs / Value Analyzer — Price Model Analysis

This notebook preserves the research layer behind the production service. It demonstrates ingestion, cleaning, EDA, data quality, normalization, feature engineering, model comparison, cross-validation, explainability, error analysis, and production export.

> **Data note:** the repository ships with a synthetic demo market dataset so the workflow is reproducible without redistributing marketplace data. Replace it with licensed market observations before interpreting accuracy as real-world pricing evidence.

## 1. Data ingestion and cleaning

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from ml.pipeline.transform import normalize_frame, engineer_features
from ml.pipeline.quality import validate_market_data
from ml.training.train import FEATURES, TARGET, candidate_models

RAW = Path('../data/raw/sample_market_listings.csv') if Path('../data/raw/sample_market_listings.csv').exists() else Path('data/raw/sample_market_listings.csv')
raw = pd.read_csv(RAW)
raw.head()

In [ ]:
raw.shape, raw.dtypes

## 2. Missing-value analysis

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False).to_frame('missing_rate')
missing

## 3. Distribution analysis

In [ ]:
raw[['asking_price','sold_price','ram_gb','storage_gb','system_age_years']].describe(percentiles=[.01,.05,.5,.95,.99])

In [ ]:
raw['sold_price'].plot(kind='hist', bins=40, title='Sold-price distribution')
plt.xlabel('Price')
plt.show()

## 4. Outlier / anomaly analysis

In [ ]:
q1, q3 = raw['sold_price'].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = raw[(raw['sold_price'] < q1 - 1.5*iqr) | (raw['sold_price'] > q3 + 1.5*iqr)]
outliers[['source_id','cpu','gpu','sold_price']].sort_values('sold_price').head(20)

## 5. Hardware normalization analysis

In [ ]:
from backend.app.services.normalization import normalize_gpu, normalize_cpu
examples = ['RTX4070', 'GeForce RTX 4070 12GB', 'NVIDIA 4070', 'Radeon RX 7900 XTX']
[(x, normalize_gpu(x).value, normalize_gpu(x).matched) for x in examples]

In [ ]:
normalized = normalize_frame(raw)
valid, rejected, quality = validate_market_data(normalized)
quality.as_dict(), rejected[['source_id','cpu','gpu','ram_gb','sold_price']].head(10)

## 6. Feature engineering

In [ ]:
training = engineer_features(valid)
training[['cpu','gpu','cpu_score','gpu_score','ram_gb','storage_gb','condition_score','sold_price']].head()

Leakage check: `asking_price` is intentionally excluded from `FEATURES`; the target is observed/sold price.

In [ ]:
FEATURES

## 7. Baseline and multiple regression models

In [ ]:
X = training[FEATURES]
y = training[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows=[]
fitted={}
for name, pipe in candidate_models().items():
    cv_mae = -cross_val_score(pipe, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        'model': name,
        'cv_mae_mean': cv_mae.mean(),
        'cv_mae_std': cv_mae.std(),
        'mae': mean_absolute_error(y_test,pred),
        'rmse': mean_squared_error(y_test,pred)**.5,
        'r2': r2_score(y_test,pred),
    })
    fitted[name]=(pipe,pred)
comparison = pd.DataFrame(rows).sort_values('cv_mae_mean')
comparison

## 8. Production model selection based on evidence

In [ ]:
winner_name = comparison.iloc[0]['model']
winner, winner_pred = fitted[winner_name]
winner_name

The selection rule is lowest mean 5-fold CV MAE, not model prestige. Holdout MAE/RMSE/R² are reported separately to check generalization.

## 9. Feature importance / explainability

In [ ]:
pre = winner.named_steps['preprocess']
model = winner.named_steps['model']
feature_names = pre.get_feature_names_out()
if hasattr(model, 'feature_importances_'):
    importance = model.feature_importances_
elif hasattr(model, 'coef_'):
    importance = np.abs(np.ravel(model.coef_))
else:
    # permutation importance can be used for models without native importances
    from sklearn.inspection import permutation_importance
    p = permutation_importance(winner, X_test, y_test, scoring='neg_mean_absolute_error', random_state=42, n_repeats=8)
    feature_names = np.array(FEATURES)
    importance = p.importances_mean
pd.DataFrame({'feature': feature_names, 'importance': importance}).sort_values('importance', ascending=False).head(20)

## 10. Error / failure analysis

In [ ]:
errors = X_test.copy()
errors['actual'] = y_test
errors['predicted'] = winner_pred
errors['error'] = errors['predicted'] - errors['actual']
errors['abs_error'] = errors['error'].abs()
errors['price_band'] = pd.cut(errors['actual'], bins=[0,800,1400,2200,10000], labels=['budget','mid','high','enthusiast'])
errors.groupby('price_band', observed=True)['abs_error'].agg(['count','mean','median'])

In [ ]:
errors.groupby('condition')['abs_error'].agg(['count','mean']).sort_values('mean', ascending=False)

In [ ]:
errors.groupby('gpu')['abs_error'].agg(['count','mean']).query('count >= 5').sort_values('mean', ascending=False).head(12)

## 11. Sensitivity to missing / unrecognized hardware

In [ ]:
probe = X_test.head(50).copy()
base = winner.predict(probe)
missing_gpu = probe.copy(); missing_gpu['gpu'] = 'UNKNOWN'; missing_gpu['gpu_score'] = 30
missing_cpu = probe.copy(); missing_cpu['cpu'] = 'UNKNOWN'; missing_cpu['cpu_score'] = 35
pd.DataFrame({
    'scenario':['recognized','GPU missing','CPU missing'],
    'mean_abs_shift':[0, np.mean(np.abs(winner.predict(missing_gpu)-base)), np.mean(np.abs(winner.predict(missing_cpu)-base))]
})

## 12. Residual diagnostics

In [ ]:
residuals = y_test.to_numpy() - winner_pred
plt.scatter(winner_pred, residuals, alpha=.5)
plt.axhline(0)
plt.xlabel('Predicted price'); plt.ylabel('Residual'); plt.title('Residuals vs prediction')
plt.show()

## 13. Export / registration contract

In [ ]:
from pathlib import Path
import joblib
artifact = Path('../backend/artifacts/price_model.joblib') if Path('../backend').exists() else Path('backend/artifacts/price_model.joblib')
# The production training script performs the actual export and optional MLflow logging.
# joblib.dump(winner, artifact)
artifact

## 14. Live market calibration and target hygiene

The production app now pools fresh **active asking / retail / open-box** observations from source adapters, but those rows are intentionally kept separate from the supervised training target. An active seller price is not a completed sale price. The structural model is still trained and evaluated against an observed outcome such as `sold_price`; live observations are short-lived calibration evidence at inference time.

The live pipeline performs provider validation, USD→CAD normalization, the same hardware normalization used by the app, extraction-quality scoring, deduplication, provenance storage, and TTL expiry. A prediction only enters hybrid mode when enough fresh, sufficiently similar comparables exist.


In [ ]:
from backend.app.config import get_settings
settings = get_settings()
{
    'live_market_enabled': settings.live_market_enabled,
    'configured_sources': settings.configured_market_sources,
    'target_currency': settings.market_currency,
    'max_comp_age_hours': settings.live_comp_max_age_hours,
    'min_similarity': settings.live_comp_min_similarity,
    'blend_cap': settings.live_comp_blend_cap,
}


### Inspecting a local live-market snapshot

When the Docker stack has refreshed provider data, the following optional cell summarizes the operational cache. It is written to fail closed when PostgreSQL is not running, so the notebook remains reproducible without API credentials or redistributed marketplace data.


In [ ]:
from sqlalchemy import create_engine

live_snapshot = pd.DataFrame()
try:
    engine = create_engine(settings.database_url)
    live_snapshot = pd.read_sql(
        '''
        SELECT source, listing_type, title, price_cad, extraction_quality,
               first_seen_at, last_seen_at, expires_at, specs_payload
        FROM live_market_listings
        WHERE active = true
        ORDER BY last_seen_at DESC
        ''',
        engine,
    )
except Exception as exc:
    print(f'Live snapshot unavailable in this notebook session: {exc}')

if not live_snapshot.empty:
    display(live_snapshot.groupby(['source', 'listing_type'])['price_cad'].agg(['count','median','min','max']))
    display(live_snapshot['extraction_quality'].describe())
else:
    print('No live rows loaded; run the market refresher with provider credentials to populate this analysis.')


### Hybrid comparable adjustment

For target system $T$ and comparable $C$, the serving layer computes a bounded hardware adjustment from the structural model:

$$\Delta_{hardware}=\hat{y}(T)-\hat{y}(C)$$

$$p_{adjusted}(C)=p_{asking}(C)+\Delta_{hardware}$$

Adjusted comparables are weighted by hardware similarity and freshness, then aggregated with a weighted median. The final prediction blends that live estimate with the structural baseline, with a hard cap on the live weight. This is designed to gain responsiveness to current market movement without allowing a handful of noisy seller asks to become the entire valuation model.

A real evaluation of this hybrid layer should compare **model-only vs hybrid** predictions on a held-out recent completed-sales dataset. Until that evaluation exists, live comparables improve recency/provenance but do not justify claiming lower real-world MAE.


## Conclusions and limitations

The notebook intentionally separates **demonstrated pipeline/model mechanics** from **real-market validity**. The checked-in supervised metrics come from the synthetic demo sold-price dataset. Live provider observations improve current-market context at serving time, but they are asking/retail evidence rather than completed-sale labels. A production accuracy claim requires licensed recent sold-price data, a held-out evaluation set, and a direct model-only versus hybrid comparison.
